### Imports

In [1]:
import json
import numpy as np
import pandas as pd
import pingouin as pg
import seaborn as sn

print(pg.__version__) # 0.5.3
print(pd.__version__) # 2.0.3
print(np.__version__) # 1.24.3
print(sn.__version__) # 0.13.0

from utils_MS import *

# %load_ext autotime

0.5.3
2.2.2
1.26.4
0.13.2


/home/ealvarez/miniconda3/envs/graph_matching/lib/python3.10/site-packages/outdated/utils.py:14: OutdatedPackageWarning: The package pingouin is out of date. Your version is 0.5.3, the latest is 0.6.1.
Set the environment variable OUTDATED_IGNORE=1 to disable these warnings.
  return warn(


In [2]:
# def run(exp):

### Parameters

In [3]:
file = open("exp.json")
experiment = json.load(file)
exp = experiment["exp"]

file = open("experiments/output/{}/parameters.json".format(exp))
params = json.load(file)

print("Exp:\t\t", exp)

data_variations = params["data_variations"]
print("Data variations:", data_variations)

apply_transformation = params["apply_transformation"]
print("Apply transformation:", apply_transformation)

threshold_corr = params["threshold_corr"]
print("Threshold corr:\t", threshold_corr)

groups_id = params["groups_id"]
print("Groups id:\t", groups_id)

subgroups_id = params["subgroups_id"]
print("Subgroups id:\t", subgroups_id)

groups_id_no = params["groups_id_no"]
print("Groups id (no):\t", groups_id_no)

Exp:		 exp9
Data variations: ['none']
Apply transformation: False
Threshold corr:	 0.5
Groups id:	 ['AR', 'CRS', 'OSA', 'LPRD', 'SGB', 'LSNB', 'RCC', 'BC', 'BPH', 'PCa', 'PD']
Subgroups id:	 {'AR': ['1', '2'], 'CRS': ['1', '2'], 'OSA': ['1', '2'], 'LPRD': ['1', '2'], 'SGB': ['1', '2'], 'LSNB': ['1', '2'], 'RCC': ['1', '2'], 'BC': ['1', '2'], 'BPH': ['1', '2'], 'PCa': ['1', '2'], 'PD': ['1', '2']}
Groups id (no):	 ['Blank', 'QC', 'Std']


In [4]:
# Remove
# groups_id = ["OSA"]

### Load dataset

In [5]:
# read raw data
df_join_raw = pd.read_csv("experiments/input/{}_raw.csv".format(exp), index_col=0)
df_join_raw

,Average Rt,Average Mz,Metabolite name,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,AR_1.7,...,PD_2.5,PD_2.6,PD_2.7,PD_2.8,PD_2.9,PD_2.10,PD_2.11,PD_2.12,PD_2.13,PD_2.14
0,1,69.99951,Unknown,0.57533,-2.21576,-1.87702,-1.51884,-1.39884,-1.29892,-1.21306,...,0.98788,-0.84910,-0.00751,0.45284,0.43153,-0.04444,0.46331,0.02780,-1.09413,-1.07540
1,1,70.04025,Unknown,2.51970,0.52452,0.86264,1.22016,1.33995,1.43969,1.52539,...,3.04923,2.96805,2.60501,3.06450,3.04323,2.56814,3.07496,2.64025,4.43489,2.05710
2,1,70.04151,Unknown,0.58098,-0.48955,-0.21483,0.07565,0.17297,0.25400,0.32363,...,1.67929,1.93973,1.11480,1.48813,1.47085,1.08484,1.49663,1.14343,0.52253,1.85662
3,1,70.04908,Unknown,2.57051,-0.10795,0.21509,0.55668,0.67112,0.76641,0.84829,...,2.38948,1.70959,1.86469,2.30371,2.28339,1.82946,2.31369,1.89836,0.67806,0.69874
4,1,70.06267,Unknown,1.15787,-0.15137,0.12754,0.29794,0.42245,0.52127,0.60354,...,2.60041,1.39557,1.36967,1.75502,1.73727,1.33853,1.76374,1.39940,1.12760,2.77683
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,1,732.79951,Unknown,2.21477,-0.58849,-0.38930,-0.17867,-0.10810,4.23613,-0.04934,...,0.97794,0.89484,0.53114,0.81967,0.80618,0.50653,0.82628,0.55450,0.10928,0.17849
5440,1,748.76437,Unknown,0.72580,-1.43132,-1.05285,-0.65267,-0.51859,-0.40695,-0.31103,...,1.16682,1.08808,0.89559,1.39809,1.37400,0.85432,1.40992,0.93504,0.20434,0.27869
5441,1,794.79590,Unknown,1.65124,-1.20428,-0.80051,-0.37358,-0.23053,-0.11143,-0.00910,...,1.03778,0.97033,1.21288,1.75795,1.73195,1.16780,1.77073,1.25592,0.32484,0.38565
5442,1,800.81295,Unknown,1.51148,-1.03648,-0.60768,-0.15429,-0.00238,0.12410,0.23278,...,1.34152,1.28640,1.55386,2.12317,2.09588,1.50710,2.13658,1.59856,0.75971,0.81518


In [6]:
# get metadata
df_join_raw_metadata = df_join_raw.iloc[:, :2]
df_join_raw_metadata

,Average Rt,Average Mz
0,1,69.99951
1,1,70.04025
2,1,70.04151
3,1,70.04908
4,1,70.06267
...,...,...
5439,1,732.79951
5440,1,748.76437
5441,1,794.79590
5442,1,800.81295


In [7]:
# filter by samples
columns_sample = [column for column in df_join_raw.columns if column.split("_")[0] not in groups_id_no]
df_join_raw_intensity = df_join_raw.loc[:, columns_sample]
df_join_raw_intensity = df_join_raw_intensity.iloc[:, 3:]
df_join_raw_intensity

,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,AR_1.7,AR_1.8,AR_1.9,AR_1.10,...,PD_2.5,PD_2.6,PD_2.7,PD_2.8,PD_2.9,PD_2.10,PD_2.11,PD_2.12,PD_2.13,PD_2.14
0,0.57533,-2.21576,-1.87702,-1.51884,-1.39884,-1.29892,-1.21306,-1.13763,-0.90208,-0.76795,...,0.98788,-0.84910,-0.00751,0.45284,0.43153,-0.04444,0.46331,0.02780,-1.09413,-1.07540
1,2.51970,0.52452,0.86264,1.22016,1.33995,1.43969,1.52539,1.60068,1.83580,1.96968,...,3.04923,2.96805,2.60501,3.06450,3.04323,2.56814,3.07496,2.64025,4.43489,2.05710
2,0.58098,-0.48955,-0.21483,0.07565,0.17297,0.25400,0.32363,0.38481,0.57584,0.68462,...,1.67929,1.93973,1.11480,1.48813,1.47085,1.08484,1.49663,1.14343,0.52253,1.85662
3,2.57051,-0.10795,0.21509,0.55668,0.67112,0.76641,0.84829,0.92023,1.14487,1.27278,...,2.38948,1.70959,1.86469,2.30371,2.28339,1.82946,2.31369,1.89836,0.67806,0.69874
4,1.15787,-0.15137,0.12754,0.29794,0.42245,0.52127,0.60354,0.67423,0.88796,1.00628,...,2.60041,1.39557,1.36967,1.75502,1.73727,1.33853,1.76374,1.39940,1.12760,2.77683
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5439,2.21477,-0.58849,-0.38930,-0.17867,-0.10810,4.23613,-0.04934,0.00114,0.12100,0.21208,...,0.97794,0.89484,0.53114,0.81967,0.80618,0.50653,0.82628,0.55450,0.10928,0.17849
5440,0.72580,-1.43132,-1.05285,-0.65267,-0.51859,-0.40695,-0.31103,-0.22674,0.03643,0.18629,...,1.16682,1.08808,0.89559,1.39809,1.37400,0.85432,1.40992,0.93504,0.20434,0.27869
5441,1.65124,-1.20428,-0.80051,-0.37358,-0.23053,-0.11143,-0.00910,0.08082,0.36159,0.47160,...,1.03778,0.97033,1.21288,1.75795,1.73195,1.16780,1.77073,1.25592,0.32484,0.38565
5442,1.51148,-1.03648,-0.60768,-0.15429,-0.00238,0.12410,0.23278,0.32828,0.62644,0.79623,...,1.34152,1.28640,1.55386,2.12317,2.09588,1.50710,2.13658,1.59856,0.75971,0.81518


In [8]:
df_join_raw_intensity.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5444 entries, 0 to 5443
Columns: 308 entries, AR_1.1 to PD_2.14
dtypes: float64(308)
memory usage: 12.8 MB


In [9]:
check_dataset(df_join_raw_intensity)

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 61931
Count zero:	 0
Count positive:	 1614821
Count greater than 1:	 1361642
Count less than -1:	 8037


### Generate graphs

In [10]:
# Transformation (log10)

if apply_transformation:
	df_join_raw_log = log10_global(df_join_raw_intensity)
else:
	df_join_raw_log = df_join_raw_intensity.copy()
df_join_raw_log.head()

,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,AR_1.7,AR_1.8,AR_1.9,AR_1.10,...,PD_2.5,PD_2.6,PD_2.7,PD_2.8,PD_2.9,PD_2.10,PD_2.11,PD_2.12,PD_2.13,PD_2.14
0,0.57533,-2.21576,-1.87702,-1.51884,-1.39884,-1.29892,-1.21306,-1.13763,-0.90208,-0.76795,...,0.98788,-0.84910,-0.00751,0.45284,0.43153,-0.04444,0.46331,0.02780,-1.09413,-1.07540
1,2.51970,0.52452,0.86264,1.22016,1.33995,1.43969,1.52539,1.60068,1.83580,1.96968,...,3.04923,2.96805,2.60501,3.06450,3.04323,2.56814,3.07496,2.64025,4.43489,2.05710
2,0.58098,-0.48955,-0.21483,0.07565,0.17297,0.25400,0.32363,0.38481,0.57584,0.68462,...,1.67929,1.93973,1.11480,1.48813,1.47085,1.08484,1.49663,1.14343,0.52253,1.85662
3,2.57051,-0.10795,0.21509,0.55668,0.67112,0.76641,0.84829,0.92023,1.14487,1.27278,...,2.38948,1.70959,1.86469,2.30371,2.28339,1.82946,2.31369,1.89836,0.67806,0.69874
4,1.15787,-0.15137,0.12754,0.29794,0.42245,0.52127,0.60354,0.67423,0.88796,1.00628,...,2.60041,1.39557,1.36967,1.75502,1.73727,1.33853,1.76374,1.39940,1.12760,2.77683


In [11]:
check_dataset(df_join_raw_log)

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 61931
Count zero:	 0
Count positive:	 1614821
Count greater than 1:	 1361642
Count less than -1:	 8037


In [12]:
# split graph in groups and subgroups

""" def split_groups_subgroups(df_join_raw_log, groups_id, subgroups_id, by_group=False):
	list_df_groups_subgroups = []
	for group in groups_id:
		df_aux = df_join_raw_log.filter(like=group)
		list_aux = []
		
		if by_group:
			list_aux.append(df_aux)
		else:
			for subgroup in subgroups_id[group]:
				list_aux.append(df_aux.filter(like="{}_{}.".format(group, subgroup)))
		list_df_groups_subgroups.append(list_aux)
	return list_df_groups_subgroups """

dict_df_groups_subgroups = split_groups_subgroups(df_join_raw_log, groups_id, subgroups_id)
dict_df_groups_subgroups[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

,AR_1.1,AR_1.2,AR_1.3,AR_1.4,AR_1.5,AR_1.6,AR_1.7,AR_1.8,AR_1.9,AR_1.10,AR_1.11,AR_1.12,AR_1.13,AR_1.14
0,0.57533,-2.21576,-1.87702,-1.51884,-1.39884,-1.29892,-1.21306,-1.13763,-0.90208,-0.76795,-0.72852,-0.69120,-0.62204,-0.58983
1,2.51970,0.52452,0.86264,1.22016,1.33995,1.43969,1.52539,1.60068,1.83580,1.96968,2.00904,2.04629,2.11532,2.14747
2,0.58098,-0.48955,-0.21483,0.07565,0.17297,0.25400,0.32363,0.38481,0.57584,0.68462,0.71660,0.74686,0.80295,0.82907
3,2.57051,-0.10795,0.21509,0.55668,0.67112,0.76641,0.84829,0.92023,1.14487,1.27278,1.31039,1.34598,1.41193,1.44265
4,1.15787,-0.15137,0.12754,0.29794,0.42245,0.52127,0.60354,0.67423,0.88796,1.00628,1.04073,1.07320,1.13309,1.16087


In [13]:
check_dataset(dict_df_groups_subgroups[groups_id[0]][subgroups_id[groups_id[0]][0]])

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 8664
Count zero:	 0
Count positive:	 67552
Count greater than 1:	 53482
Count less than -1:	 1907


In [14]:
# No apply transpose for kneighbors_graph
# dict_groups_subgroups_t = dict_df_groups_subgroups.copy()

# Aplly Transpose
dict_groups_subgroups_t = transpose_global(dict_df_groups_subgroups)

dict_groups_subgroups_t[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

,0,1,2,3,4,5,6,7,8,9,...,5434,5435,5436,5437,5438,5439,5440,5441,5442,5443
0,0.57533,2.51970,0.58098,2.57051,1.15787,1.27213,2.62029,2.63990,2.65040,2.73757,...,1.64684,1.98987,-0.30933,1.23702,1.87136,2.21477,0.72580,1.65124,1.51148,1.94236
1,-2.21576,0.52452,-0.48955,-0.10795,-0.15137,0.10341,0.38295,0.46580,0.41341,0.57550,...,0.05434,-0.07692,-2.00218,-0.30250,-0.26303,-0.58849,-1.43132,-1.20428,-1.03648,-1.79295
2,-1.87702,0.86264,-0.21483,0.21509,0.12754,0.36245,0.63531,0.71463,0.67240,0.81397,...,0.42073,0.22067,-1.63513,0.01170,-0.03679,-0.38930,-1.05285,-0.80051,-0.60768,-1.44649
3,-1.51884,1.22016,0.07565,0.55668,0.29794,0.63636,0.90215,0.97773,0.94624,1.06613,...,0.80815,0.53533,-1.24701,0.34392,0.20243,-0.17867,-0.65267,-0.37358,-0.15429,6.03212
4,-1.39884,1.33995,0.17297,0.67112,0.42245,0.72813,0.99156,1.06588,1.03799,1.15061,...,0.93795,0.64076,-1.11697,0.45524,0.28258,-0.10810,-0.51859,-0.23053,-0.00238,-1.08016


In [15]:
check_dataset(dict_groups_subgroups_t[groups_id[0]][subgroups_id[groups_id[0]][0]])

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 8664
Count zero:	 0
Count positive:	 67552
Count greater than 1:	 53482
Count less than -1:	 1907


In [16]:
from sklearn.preprocessing import StandardScaler
from sklearn.covariance import LedoitWolf
from sklearn.neighbors import kneighbors_graph

def correlation_ledoitwolf_global(exp, dict_groups_subgroups_t):
	dict_groups_subgroups_t_corr = {}
	for group_id, dict_groups in dict_groups_subgroups_t.items():
		dict_aux = {}
		for subgroup_id, df_subgroup in dict_groups.items():
			print(group_id, subgroup_id, df_subgroup.shape)
			
			""" import numpy as np
			cov = np.cov(df_subgroup.values, rowvar=False)
			cond = np.linalg.cond(cov)
			print("Condition number:", cond, cond > 1e8) # ill-conditioned if > 1e8 (True, instable) """

			scaler = StandardScaler()
			df_subgroup_scaled = scaler.fit_transform(df_subgroup)
			lw = LedoitWolf()
			lw.fit(df_subgroup_scaled)

			# Matriz de covarianza regularizada
			cov = lw.covariance_
			std = np.sqrt(np.diag(cov))
			corr = cov / np.outer(std, std)
			matrix = pd.DataFrame(corr)
		
			dict_aux[subgroup_id] = matrix
			
			matrix.to_csv("experiments/output/{}/correlations/{}_{}.csv".format(exp, group_id, subgroup_id), index=True)
		dict_groups_subgroups_t_corr[group_id] = dict_aux
	return dict_groups_subgroups_t_corr

def correlation_kneighbors_graph_global(exp, dict_groups_subgroups_t):
	dict_groups_subgroups_t_corr = {}
	for group_id, dict_groups in dict_groups_subgroups_t.items():
		dict_aux = {}
		for subgroup_id, df_subgroup in dict_groups.items():
			print(group_id, subgroup_id, df_subgroup.shape)
			
			""" import numpy as np
			cov = np.cov(df_subgroup.values, rowvar=False)
			cond = np.linalg.cond(cov)
			print("Condition number:", cond, cond > 1e8) # ill-conditioned if > 1e8 (True, instable) """
			
			k = 10
			scaler = StandardScaler()
			df_subgroup_scaled = scaler.fit_transform(df_subgroup)
			A = kneighbors_graph(
				df_subgroup_scaled, # X, X_scaled
				n_neighbors=k,
				metric="euclidean", # "cosine",
				mode="distance",
				include_self=True
			)
			matrix = pd.DataFrame(A.toarray())
		
			dict_aux[subgroup_id] = matrix
			
			matrix.to_csv("experiments/output/{}/correlations/{}_{}.csv".format(exp, group_id, subgroup_id), index=True)
		dict_groups_subgroups_t_corr[group_id] = dict_aux
	return dict_groups_subgroups_t_corr

In [17]:
# Correlation matrix (partial correlation)

# Option 1
# dict_groups_subgroups_t_corr = correlation_global(exp, dict_groups_subgroups_t)

# Option 2
dict_groups_subgroups_t_corr = correlation_ledoitwolf_global(exp, dict_groups_subgroups_t)

# Option 3
# dict_groups_subgroups_t_corr = correlation_kneighbors_graph_global(exp, dict_groups_subgroups_t)

dict_groups_subgroups_t_corr[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

AR 1 (14, 5444)


AR 2 (14, 5444)
CRS 1 (14, 5444)
CRS 2 (14, 5444)
OSA 1 (14, 5444)
OSA 2 (14, 5444)
LPRD 1 (14, 5444)
LPRD 2 (14, 5444)
SGB 1 (14, 5444)
SGB 2 (14, 5444)
LSNB 1 (14, 5444)
LSNB 2 (14, 5444)
RCC 1 (14, 5444)
RCC 2 (14, 5444)
BC 1 (14, 5444)
BC 2 (14, 5444)
BPH 1 (14, 5444)
BPH 2 (14, 5444)
PCa 1 (14, 5444)
PCa 2 (14, 5444)
PD 1 (14, 5444)
PD 2 (14, 5444)


,0,1,2,3,4,5,6,7,8,9,...,5434,5435,5436,5437,5438,5439,5440,5441,5442,5443
0,1.000000,0.483011,0.409727,0.503280,0.451822,0.443424,0.502444,0.502741,0.502932,0.270128,...,0.432015,0.274301,0.443558,0.225076,0.499472,0.199005,0.478539,0.500200,0.483861,0.116111
1,0.483011,1.000000,0.475342,0.482293,0.494763,0.492350,0.474019,0.475901,0.477357,0.271608,...,0.487156,0.299942,0.492509,0.268724,0.462120,0.186330,0.503059,0.495426,0.503278,0.085012
2,0.409727,0.475342,1.000000,0.408250,0.494198,0.499032,0.392128,0.395670,0.398457,0.245460,...,0.501639,0.299142,0.499203,0.292187,0.371011,0.152380,0.480102,0.438994,0.474329,0.039925
3,0.503280,0.482293,0.408250,1.000000,0.450721,0.442221,0.502584,0.502852,0.503020,0.269904,...,0.430710,0.273640,0.442355,0.224128,0.499774,0.199086,0.477748,0.499918,0.483158,0.116583
4,0.451822,0.494763,0.494198,0.450721,1.000000,0.500202,0.438474,0.441198,0.443328,0.263049,...,0.499471,0.307506,0.501023,0.289025,0.421492,0.165764,0.497002,0.472223,0.494256,0.038880


In [18]:
# Check correlation matrices

dict_groups_subgroups_t_corr

{'AR': {'1':           0         1         2         3         4         5         6     \
  0     1.000000  0.483011  0.409727  0.503280  0.451822  0.443424  0.502444   
  1     0.483011  1.000000  0.475342  0.482293  0.494763  0.492350  0.474019   
  2     0.409727  0.475342  1.000000  0.408250  0.494198  0.499032  0.392128   
  3     0.503280  0.482293  0.408250  1.000000  0.450721  0.442221  0.502584   
  4     0.451822  0.494763  0.494198  0.450721  1.000000  0.500202  0.438474   
  ...        ...       ...       ...       ...       ...       ...       ...   
  5439  0.199005  0.186330  0.152380  0.199086  0.165764  0.169879  0.199631   
  5440  0.478539  0.503059  0.480102  0.477748  0.497002  0.495242  0.468717   
  5441  0.500200  0.495426  0.438994  0.499918  0.472223  0.466655  0.496195   
  5442  0.483861  0.503278  0.474329  0.483158  0.494256  0.491712  0.475038   
  5443  0.116111  0.085012  0.039925  0.116583  0.038880  0.059596  0.121355   
  
            7         8   

In [19]:
check_dataset(dict_groups_subgroups_t_corr[groups_id[0]][subgroups_id[groups_id[0]][1]])

Checking dataset
Count infinite:	 0
Count nan:	 0
Count negative:	 14360024
Count zero:	 0
Count positive:	 15277112
Count greater than 1:	 286
Count less than -1:	 0


In [20]:
def build_graph_weight_global_directed_new(exp, dict_groups_subgroups_t_corr, threshold=0.3):
	dict_groups_subgroups_t_corr_g = {}
	for group_id, dict_groups in dict_groups_subgroups_t_corr.items():
		dict_aux = {}
		for subgroup_id, df_subgroup in dict_groups.items():
			# Percentil Correlaciones conservadas 	Densidad esperadas
			# 90		10 %						Alta
			# 95		5 %							Media
			# 97.5		2.5 %						Baja
			# 99		1 %							Muy baja
			threshold = np.percentile(np.abs(df_subgroup), 95)
			
			df_weighted_edges = (df_subgroup.where(np.triu(np.ones(df_subgroup.shape), k=1).astype(bool)).stack())
			df_weighted_edges = df_weighted_edges.dropna().to_frame()
			df_weighted_edges.reset_index(inplace=True)
			df_weighted_edges.columns = ["source", "target", "weight"]
			df_weighted_edges = df_weighted_edges[df_weighted_edges["weight"].abs() >= threshold]
			df_weighted_edges["subgroup"] = [subgroup_id] * len(df_weighted_edges)
			dict_aux[subgroup_id] = df_weighted_edges
			
			df_weighted_edges.to_csv("experiments/output/{}/preprocessing/edges/{}_{}.csv".format(exp, group_id, subgroup_id), index=False)
			# G = nx.from_pandas_edgelist(df_weighted_edges, "source", "target", edge_attr=["weight"])
			# print(groups_id[i], subgroups_id[groups_id[i]][j], G.number_of_nodes(), G.number_of_edges())
			# nx.write_gexf(G, "experiments/output/{}/preprocessing/graphs/graphs_{}_{}.gexf".format(exp, groups_id[i], subgroups_id[groups_id[i]][j]))
		dict_groups_subgroups_t_corr_g[group_id] = dict_aux
	return dict_groups_subgroups_t_corr_g

In [21]:
# Build graph (corpus graphs)

# dict_groups_subgroups_t_corr_g = build_graph_weight_global_directed(exp, dict_groups_subgroups_t_corr, threshold=threshold_corr)
dict_groups_subgroups_t_corr_g = build_graph_weight_global_directed_new(exp, dict_groups_subgroups_t_corr, threshold=threshold_corr)
dict_groups_subgroups_t_corr_g[groups_id[0]][subgroups_id[groups_id[0]][0]].head()

,source,target,weight,subgroup
0,0,1,0.483011,1
1,0,2,0.409727,1
2,0,3,0.503280,1
3,0,4,0.451822,1
4,0,5,0.443424,1


In [22]:
def create_graph_data_directed_features(exp, groups_id, subgroups_id, dict_df_groups_subgroups, df_join_raw_metadata):	
	for group_id in tqdm(groups_id):
		for subgroup_id in tqdm(subgroups_id[group_id]):
			df_weighted_edges = pd.read_csv("experiments/output/{}/preprocessing/edges/{}_{}.csv".format(exp, group_id, subgroup_id))
			# print(df_weighted_edges)
			G = nx.from_pandas_edgelist(df_weighted_edges, "source", "target", edge_attr=["weight", "subgroup"])
			dict_id_idx = dict(zip(list(G.nodes()), range(G.number_of_nodes())))
			G = nx.relabel_nodes(G, dict_id_idx)

			df_nodes = dict_df_groups_subgroups[group_id][subgroup_id].loc[list(dict_id_idx.keys())] # A_1.1, A_1.2, A_1.3
			# from IPython.display import display
			# display(df_nodes)

			# nodes, with node features
			metadata = df_join_raw_metadata.loc[df_nodes.index] # Average Rt, Average Mz
			# intensity = df_join_raw_log.loc[df_nodes.index] # A_1.1, A_1.2, A_1.3, A_2.1, ...

			""" e = 1e-8
			mz = metadata.iloc[:, 1].values
			rt = metadata.iloc[:, 0].values
			intensity_mean = df_nodes.mean(axis=1).values
			intensity_std = df_nodes.std(axis=1).values
			intensity_cv = intensity_std / intensity_mean
			presence_ratio = (df_nodes > 0).mean(axis=1)

			mz_log = np.log10(mz + e)
			# intensity_mean_log = np.log10(intensity_mean + e)
			
			# mz_z = (mz_log - mz_log.mean()) / mz_log.std() # z-score
			rt_z = (rt - rt.mean()) / rt.std() # z-score
			# intensity_mean_z = (intensity_mean_log - intensity_mean_log.mean()) / intensity_mean_log.std() # z-score

			data_node = {
				"idx": list(dict_id_idx.values()),
				"id": list(dict_id_idx.keys()),
				"mz": mz_log,
				"rt": rt_z,
				"intensity_mean": intensity_mean,
				"intensity_std": intensity_std,
				"intensity_cv": intensity_cv,
				"presence_ratio": presence_ratio
			}
			for i in range(len(df_nodes.columns)):
				data_node[i] = df_nodes.iloc[:, i]

			df_node_features = pd.DataFrame(data_node)
			# df_node_features.insert(0, "idx", list(dict_id_idx.values()))
			# df_node_features.insert(1, "id", list(dict_id_idx.keys()))
			df_node_features.to_csv("experiments/output/{}/preprocessing/graphs_data/nodes_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)
			# print(df_node_features) """

			data_node = {
				"idx": list(dict_id_idx.values()),
				"id": list(dict_id_idx.keys()),
				"mz": metadata.iloc[:, 1].values,
				"rt": metadata.iloc[:, 0].values,
			}
			for i in range(len(df_nodes.columns)):
				data_node[i] = df_nodes.iloc[:, i]

			df_node_features = pd.DataFrame(data_node)
			df_node_features.to_csv("experiments/output/{}/preprocessing/graphs_data/nodes_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)
			# print(df_node_features)

			# edges
			edges = list(G.edges())
			df_edges = pd.DataFrame(edges, columns=["source", "target"])
			df_edges["weight"] = [G.get_edge_data(*edge)["weight"] for edge in edges]
			df_edges["subgroup"] = [G.get_edge_data(*edge)["subgroup"] for edge in edges]
			df_edges.to_csv("experiments/output/{}/preprocessing/graphs_data/edges_{}_{}.csv".format(exp, group_id, subgroup_id), index=False)


In [23]:
# create dataset - nodes/edge data for PyTorch Geometric/DGL framework

# IMPORTANT
dict_df_groups_subgroups_ = split_groups_subgroups(df_join_raw_intensity, groups_id, subgroups_id) # Important (intesities without Log)

for data_variation in data_variations:
	if data_variation == "none":
		create_graph_data_directed_features(exp, groups_id, subgroups_id, dict_df_groups_subgroups_, df_join_raw_metadata)
	else:
		# dynamic graph to static graph
		create_graph_data_directed_variation(exp, groups_id, subgroups_id, dict_df_groups_subgroups_, data_variation)

100%|██████████| 11/11 [01:18<00:00,  7.15s/it]


In [24]:
# details
list_details = []
	
for group_id in groups_id:
	subgroups_id_ = []
	for data_variation in data_variations:
		if data_variation == "none":
			subgroups_id_ += subgroups_id[group_id]
		else:
			subgroups_id_ += [data_variation]
	# print(subgroups)
	
	for subgroup_id_ in subgroups_id_:
		try:
			df_edges = pd.read_csv("experiments/output/{}/preprocessing/graphs_data/edges_{}_{}.csv".format(exp, group_id, subgroup_id_))

			G = nx.from_pandas_edgelist(df_edges.iloc[:, [0, 1]])
			list_details.append([group_id, subgroup_id_, G.number_of_nodes(), G.number_of_edges(), nx.density(G), np.nan, nx.is_connected(G)])
		except:
			list_details.append([group_id, subgroup_id_, G.number_of_nodes(), G.number_of_edges(), np.nan, np.nan, np.nan])

df_details = pd.DataFrame(list_details, columns=["Group", "Subgroup", "Num. nodes", "Num. edges", "Density", "Diameter", "Is connected"])
df_details.to_csv("experiments/output/{}/preprocessing/graphs_data/summary.csv".format(exp), index=False)

df_details = pd.read_csv("experiments/output/{}/preprocessing/graphs_data/summary.csv".format(exp))
df_details

,Group,Subgroup,Num. nodes,Num. edges,Density,Diameter,Is connected
0,AR,1,5382,738207,0.050980,NaN,False
1,AR,2,5433,738207,0.050027,NaN,True
2,CRS,1,5444,738207,0.049826,NaN,True
3,CRS,2,5444,738207,0.049826,NaN,True
4,OSA,1,5444,738207,0.049826,NaN,True
5,OSA,2,5444,738207,0.049826,NaN,True
6,LPRD,1,5444,738207,0.049826,NaN,True
7,LPRD,2,5444,738207,0.049826,NaN,True
8,SGB,1,5444,738207,0.049826,NaN,True
9,SGB,2,5444,738207,0.049826,NaN,True
